# 第45章 生产量化系统工程基础 (Chapter 45 Production Quant)

本 Notebook 演示离线生产级量化流水线（Production Pipeline）的核心执行与工程闸门逻辑。

### 核心设计原则
1. **单向因果与 PIT 隔离**：决策观测严格满足 $prior\_event\_time < event\_time \le as\_of \le signal\_time < trade\_time$，未来数据与撮合价格严格隔离。
2. **全生命周期质量闸门**：类型、单位、时效、陈旧阈值、NaN/Inf/Bool 检查在任何成交发生前阻断。
3. **确定性商业产物**：业务摘要、成交流水、账本记录完全确定；运行时信息（耗时、PID 等）独立切分。
4. **SQLite 事务状态与幂等恢复**：Run Key 绑定指纹，唯一 `fill_id`，崩溃恢复无重复成交、无二次支付。

R2限定返修：执行与估值资产集合完整覆盖；一般压力情景简单收益有限且≥−1（正冲击允许，损失可为负）；动量计算后也要有限。固定配置允许正实数股数，未验证券商lot size；非正信号目标为0。金额对账容差为1e−6 USD。故障按实际累计成交次序，STOPPED不可同key恢复新交易。成熟标签可在到期后在线更新监控，本Notebook无训练/漂移部署。


本 Notebook 使用离线教学构造数据，或明确标注的历史结果。新运行输出保存在单独的 `runs/` 子目录；这些内容不构成真实市场研究。


In [ ]:
import os, sys
from pathlib import Path
from uuid import uuid4
candidate = Path.cwd().resolve()
ROOT = next((p for p in (candidate, *candidate.parents)
             if (p / "pyproject.toml").is_file() and (p / "src/quant_course").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Start from the reader package root or a subdirectory")
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "code"))
os.environ["MPLBACKEND"] = "Agg"
os.chdir(ROOT)
RUN_DIR = ROOT / "runs" / ("ch45_production_quant_" + uuid4().hex)
RUN_DIR.mkdir(parents=True, exist_ok=False)
workspace_root = ROOT
print("Run directory:", RUN_DIR.relative_to(ROOT))
import csv, json
import matplotlib.pyplot as plt
from quant_course.production_pipeline import default_config, constructed_inputs, run_pipeline


## 1. 检查默认配置与构造输入

流水线输入显式声明货币、单位、版本以及观测与未来撮合的严格切分。

In [ ]:
cfg = default_config()
inputs = constructed_inputs()

print("Initial Cash:", cfg["initial_cash"], cfg["currency"])
print("Half Spread (bps):", cfg["execution"]["half_spread_bps"])
print("Commission Rate:", cfg["execution"]["commission_rate"])
print("Securities metadata:", inputs["metadata"]["securities"])
print("Observations count:", len(inputs["observations"]))
print("Signal Time:", inputs["metadata"]["signal_time"])

## 2. 运行生产流水线

调用核心 `run_pipeline` 执行完整流程：
- 校验配置与输入数据闸门（时效、未来时点、单位、数值有效性）
- 特征生成（单期动量 $momentum > 0$）与保守事前杠杆及压力测试
- SQLite 状态绑定、逐笔持久化与执行账本撮合
- 账本与独立方程对账（实际残差与容差检验）
- 原子暂存发布与校验
未发布可新staging恢复；已发布且DB锚定hash有效可补COMPLETED；已发布损坏显式拒绝并保留现场。绘图是核心发布后的附属工件，不在同一事务中。


In [ ]:
run_dir = RUN_DIR
state_db, output_dir = run_dir / "state.db", run_dir / "artifacts"
summary = run_pipeline(config=cfg, inputs=inputs, run_key="nb_demo_run",
                       state_db=state_db, output_dir=output_dir)
print(json.dumps(summary, indent=2))
print("Preserved run:", run_dir.relative_to(ROOT))


## 3. 结果核验与独立平衡表对账

验证各项关键指标与容差残差：
- 最终现金：$7995.998$ USD
- 最终净值：$9995.998$ USD
- 交易成本：点差 $2.000$ + 佣金 $2.002 = 4.002$ USD
- 毛收益 $0.000$ USD，净收益 $-4.002$ USD
- 平衡表对账状态：PASS（实际残差在声明浮点容差范围内）

In [ ]:
assert abs(summary["final_cash"] - 7995.998) <= 1e-6
assert abs(summary["terminal_nav"] - 9995.998) <= 1e-6
assert abs(summary["net_pnl"] + 4.002) <= 1e-6
assert summary["reconciliation"]["status"] == "PASS"
assert summary["reconciliation"]["cash_reconciled"] is True
assert summary["reconciliation"]["positions_reconciled"] is True
assert summary["reconciliation"]["nav_pnl_reconciled"] is True
assert summary["gate_outcomes"]["balance_sheet_reconciliation"] == "PASS"
print("CONSTRUCTED baseline agrees with the hand calculation; author candidate only.")

## 4. 可视化：动态多期现金与净值演进

从落地的 `ledger_events.csv` 动态提取各阶段真实状态并绘图。

In [ ]:
ledger_csv_path = output_dir / "ledger_events.csv"
stages = []
cash_series = []
nav_series = []

if ledger_csv_path.exists():
    with open(ledger_csv_path, "r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            evt = row.get("event_type", "EVENT")
            ts = row.get("timestamp", "")
            short_time = ts[11:16] if len(ts) >= 16 else ts
            stages.append(f"{evt}\n({short_time})")
            cash_series.append(float(row.get("cash_after", 0.0)))
            nav_series.append(float(row.get("nav", 0.0)))

if not stages:
    stages = ["Initial", "Final"]
    cash_series = [summary["initial_cash"], summary["final_cash"]]
    nav_series = [summary["initial_cash"], summary["terminal_nav"]]

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(stages, nav_series, marker="o", color="#2980b9", linewidth=2, label="NAV (USD)")
ax.plot(stages, cash_series, marker="s", color="#e67e22", linewidth=2, linestyle="--", label="Cash (USD)")
ax.set_ylabel("USD")
ax.set_title("CONSTRUCTED: Cash & NAV by recorded event", fontweight="bold")
ax.grid(True, linestyle=":", alpha=0.6)
ax.legend(loc="center right")
plt.tight_layout()
plt.savefig(RUN_DIR / "ch45_production_quant_cell09.png", dpi=150, bbox_inches="tight"); plt.show(); plt.close()